# 02 · Data preparation → silver

Bronze is an untouchable copy of what was in the operational system. **Silver is
the first layer where decisions get made**, and each one is argued here before it
is written in code.

### The rule that governs this notebook

Silver holds **deterministic** features only — a cut on age, a boolean for zero
balance. Nothing here learns a statistic from the data.

Anything that *does* learn — scaling, encoding, imputation — lives inside the
sklearn `Pipeline` in notebook 04, which is fitted **after** the train/test split.
That separation is the anti-leakage control of the project. If a scaler were fitted
here, it would have seen the test rows, and every number reported later would be
quietly optimistic.

In [2]:
import sys
sys.path.append("../src")

from config import bootstrap

ctx = bootstrap()
spark, w = ctx.spark, ctx.w

connected to Databricks
  branch   : sandbox
  catalog  : bank_churn_eng
  identity : juzoushio@...


In [3]:
import pandas as pd
from pyspark.sql import functions as F
from config import UC_CATALOG

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

BRONZE_TABLE = f"{UC_CATALOG}.bronze.customers_raw"
SILVER_TABLE = f"{UC_CATALOG}.silver.customers_clean"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.silver")

bronze = spark.table(BRONZE_TABLE)
print("rows in bronze:", f"{bronze.count():,}")

# A pandas copy for diagnosis. DECIMAL columns arrive as Decimal objects and
# have to be cast before any arithmetic works on them.
df = bronze.toPandas()
for c in ["balance", "estimated_salary"]:
    df[c] = df[c].astype(float)

META = [c for c in ("_ingested_at", "_source_file", "_source_url", "_extracted_at")
        if c in df.columns]
print("traceability columns excluded from analysis:", META)

rows in bronze: 10,000
traceability columns excluded from analysis: ['_source_file', '_source_url', '_extracted_at']


## 1 · Quality profile — the "before"

Taken on bronze, unmodified. Section 6 repeats it on silver so the two can be
compared.

In [4]:
analysis_cols = [c for c in df.columns if c not in META]

profile_before = pd.DataFrame({
    "dtype":       df[analysis_cols].dtypes.astype(str),
    "n_null":      df[analysis_cols].isna().sum(),
    "n_unique":    df[analysis_cols].nunique(),
    "cardinality": (df[analysis_cols].nunique() / len(df) * 100).round(2),
})
profile_before

,dtype,n_null,n_unique,cardinality
row_number,int32,0,10000,100.00
customer_id,int32,0,10000,100.00
surname,object,0,2932,29.32
credit_score,int32,0,460,4.60
geography,object,0,3,0.03
gender,object,0,2,0.02
age,int32,0,70,0.70
tenure,int32,0,11,0.11
balance,float64,0,6382,63.82
num_of_products,int32,0,4,0.04


In [5]:
num_cols = ["credit_score", "age", "tenure", "balance",
            "num_of_products", "estimated_salary"]

desc = df[num_cols].describe().T
desc["skew"]     = df[num_cols].skew()
desc["kurtosis"] = df[num_cols].kurtosis()

print(desc.round(2).to_string())
print()
print("Look at estimated_salary and tenure: skew near 0 and kurtosis near -1.2.")
print("Those are the exact values of a uniform distribution. Real salaries pile up")
print("in the low bands with a long right tail. Second sign that this is synthetic.")

                    count       mean       std     min       25%        50%        75%        max  skew  kurtosis
credit_score      10000.0     650.53     96.65  350.00    584.00     652.00     718.00     850.00 -0.07     -0.43
age               10000.0      38.92     10.49   18.00     32.00      37.00      44.00      92.00  1.01      1.40
tenure            10000.0       5.01      2.89    0.00      3.00       5.00       7.00      10.00  0.01     -1.17
balance           10000.0   76485.89  62397.41    0.00      0.00   97198.54  127644.24  250898.09 -0.14     -1.49
num_of_products   10000.0       1.53      0.58    1.00      1.00       1.00       2.00       4.00  0.75      0.58
estimated_salary  10000.0  100090.24  57510.49   11.58  51002.11  100193.92  149388.25  199992.48  0.00     -1.18

Look at estimated_salary and tenure: skew near 0 and kurtosis near -1.2.
Those are the exact values of a uniform distribution. Real salaries pile up
in the low bands with a long right tail. Second sign

---

## 2 · Decision 1 — what is `balance = 0`?

The first substantive decision, and it has no obvious answer. Three readings are
possible and each leads to different treatment:

**(a) A legitimate value.** The customer genuinely holds no balance.
**(b) A missing value encoded as zero.** Common in exported data.
**(c) A different product state.** Not a current account at all.

The treatment differs completely: leave it, impute it, or flag it. So it gets
investigated rather than assumed.

In [6]:
zero = df["balance"] == 0

print(f"balance = 0 : {zero.sum():,} ({zero.mean():.1%})")
print(f"churn if balance = 0 : {df.loc[zero,  'exited'].mean():.1%}")
print(f"churn if balance > 0 : {df.loc[~zero, 'exited'].mean():.1%}")

print("\nzero balance by country:")
print(df.assign(z=zero).groupby("geography")["z"].mean().mul(100).round(1).to_string())

print("\ndistribution of balance when it is > 0:")
print(df.loc[~zero, "balance"].describe().round(0).to_string())

balance = 0 : 3,617 (36.2%)
churn if balance = 0 : 13.8%
churn if balance > 0 : 24.1%

zero balance by country:
geography
France     48.2
Germany     0.0
Spain      48.4

distribution of balance when it is > 0:
count      6383.0
mean     119827.0
std       30095.0
min        3769.0
25%      100182.0
50%      119840.0
75%      139512.0
max      250898.0


The first result looks conclusive: 13.8% churn with zero balance against 24.1% with
a balance. But **Germany has not one single customer with zero balance**, and
Germany churns at 32.4%.

That means every German sits in the "balance > 0" bucket, dragging it upward. The
comparison is contaminated by country. The honest move is to remove Germany and
look again.

In [7]:
no_de = df[df["geography"] != "Germany"].copy()
z = no_de["balance"] == 0

print("-- France and Spain only --")
print(f"balance = 0 : {z.sum():5,}  churn {no_de.loc[z,  'exited'].mean():.1%}")
print(f"balance > 0 : {(~z).sum():5,}  churn {no_de.loc[~z, 'exited'].mean():.1%}")

print("\n-- churn crossing country and balance state --")
print(df.assign(z=df["balance"] == 0)
        .pivot_table(index="geography", columns="z", values="exited",
                     aggfunc=["mean", "size"]).round(3).to_string())

-- France and Spain only --
balance = 0 : 3,617  churn 13.8%
balance > 0 : 3,874  churn 18.7%

-- churn crossing country and balance state --
            mean           size        
z          False  True    False   True 
geography                              
France     0.182  0.139  2596.0  2418.0
Germany    0.324    NaN  2509.0     NaN
Spain      0.196  0.136  1278.0  1199.0


### The effect is real, and half the size it looked

| | apparent | isolated |
|---|---|---|
| balance = 0 | 13.8% | 13.8% |
| balance > 0 | **24.1%** | **18.7%** |
| difference | 10.3 pp | **4.9 pp** |

A confounder cut the effect in half. Reporting the 10.3 would not have been a lie,
but it would have been wrong — and nobody would have caught it without looking at
the country breakdown.

### Does the level of balance matter, or only the zero?

If among customers who have a balance the amount does not discriminate, then
`balance_zero` captures all the information and the continuous variable adds
nothing.

In [8]:
with_balance = df[df["balance"] > 0].copy()
with_balance["q"] = pd.qcut(with_balance["balance"], 5, labels=["Q1","Q2","Q3","Q4","Q5"])

print("churn by balance quintile (balance > 0 only):")
print(with_balance.groupby("q", observed=True)["exited"]
      .agg(churn=lambda s: round(s.mean()*100, 1), n="size").to_string())

print("\ncountry profile side by side:")
print(df.groupby("geography").agg(
    mean_age=("age", "mean"),
    pct_active=("is_active_member", "mean"),
    mean_products=("num_of_products", "mean"),
    mean_balance=("balance", "mean"),
    churn=("exited", "mean"),
).round(3).to_string())

churn by balance quintile (balance > 0 only):
    churn     n
q              
Q1   20.8  1277
Q2   25.2  1276
Q3   26.8  1277
Q4   24.8  1276
Q5   22.9  1277

country profile side by side:
           mean_age  pct_active  mean_products  mean_balance  churn
geography                                                          
France       38.512       0.517          1.531     62092.637  0.162
Germany      39.772       0.497          1.520    119730.116  0.324
Spain        38.891       0.530          1.539     61818.148  0.167


### Decision 1 · `balance = 0` is a legitimate and distinct state

The evidence that settles it:

- **A gap in the distribution.** There is no balance between 0 and about 3,769.
  A null encoded as zero does not produce a jump like that; a point mass at exactly
  zero with empty space above it is a different *state*, not a missing number.
- **The effect survives the confounder.** Smaller, but it holds at 4.9 points once
  Germany is removed.
- **The level barely discriminates** among those who do have a balance.

**Treatment:** keep `balance` as it is, and add a boolean `balance_zero`. Nothing
is imputed. Imputing here would invent a balance for 36% of the customers and
destroy the very signal that was just measured.

---

## 3 · Sensitive variables

`gender` and `geography` have to be examined from a bias standpoint and their use
justified. Geography already predicts strongly with no explanation. Gender is next.

In [9]:
print("-- gender --")
print(df.groupby("gender").agg(
    n=("exited", "size"),
    churn=("exited", lambda s: round(s.mean()*100, 1)),
    mean_age=("age", "mean"),
    pct_active=("is_active_member", "mean"),
    products=("num_of_products", "mean"),
).round(3).to_string())

print("\n-- gender within each country --")
print(df.pivot_table(index="geography", columns="gender",
                     values="exited", aggfunc="mean").round(3).to_string())

print("\n-- activity, the one variable the bank can act on --")
print(df.groupby("is_active_member")["exited"]
        .agg(churn=lambda s: round(s.mean()*100, 1), n="size").to_string())

-- gender --
           n  churn  mean_age  pct_active  products
gender                                             
Female  4543   25.1    39.238       0.503     1.544
Male    5457   16.5    38.658       0.525     1.519

-- gender within each country --
gender     Female   Male
geography               
France      0.203  0.127
Germany     0.376  0.278
Spain       0.212  0.131

-- activity, the one variable the bank can act on --
                  churn     n
is_active_member             
0                  26.9  4849
1                  14.3  5151


### Gender has the same shape as Germany

25.1% against 16.5%, a gap of more than ten standard errors. And again age,
activity and products are practically identical: **no observable variable explains
the difference**. It replicates in all three countries, so it is not a country
effect in disguise.

That is exactly the situation where a variable is dangerous. It predicts well
precisely because it stands in for something unmeasured, and using it means
allocating a benefit on the basis of a protected characteristic without being able
to say why it works.

**The decision is not taken here.** Notebook 05 measures what excluding it costs,
and the decision is made against that number. Deciding now, without the cost, would
be a posture rather than a choice.

---

## 4 · The remaining variables

`credit_score` has not been looked at, and `num_of_products` is worth measuring —
the balance pattern hints that it is strong.

In [10]:
df["_cs"] = pd.qcut(df["credit_score"], 5, labels=["Q1","Q2","Q3","Q4","Q5"])
print("churn by credit_score quintile:")
print(df.groupby("_cs", observed=True)["exited"]
        .agg(churn=lambda s: round(s.mean()*100, 1), n="size").to_string())
df.drop(columns="_cs", inplace=True)

print("\nSpearman correlation with exited:")
for c in ["credit_score", "estimated_salary", "balance", "num_of_products", "age"]:
    print(f"  {c:<18} {df[c].corr(df['exited'], method='spearman'):+.4f}")

print("\nchurn by number of products:")
print(df.groupby("num_of_products")["exited"]
        .agg(churn=lambda s: round(s.mean()*100, 1), n="size").to_string())

churn by credit_score quintile:
     churn     n
_cs             
Q1    22.5  2010
Q2    20.8  2020
Q3    19.7  2010
Q4    18.3  1981
Q5    20.5  1979

Spearman correlation with exited:
  credit_score       -0.0233
  estimated_salary   +0.0121
  balance            +0.1111
  num_of_products    -0.1253
  age                +0.3240

churn by number of products:
                 churn     n
num_of_products             
1                 27.7  5084
2                  7.6  4590
3                 82.7   266
4                100.0    60


### The methodological finding of the project

```
1 product   →  27.7%   (n = 5,084)
2 products  →   7.6%   (n = 4,590)
3 products  →  82.7%   (n =   266)
4 products  → 100.0%   (n =    60)
```

**All sixty customers with four products left.** Not 95%, not 98%. Every single one.

Human behaviour does not produce a clean 100% across sixty cases. A rule does. This
is the third independent sign that the dataset is synthetic, and it carries a
practical consequence: any model will learn that rule perfectly and look brilliant
doing it, on a pattern that would not exist in real data.

It also breaks monotonicity — risk falls from 1 to 2 products and then explodes.
A linear model cannot represent that without being told; a tree finds it on its own.
That is already an argument about model choice, made before any model was trained.

---

## 5 · Transformations

### What is dropped

| Column | Why |
|---|---|
| `row_number` | Technical identifier, 100% cardinality, zero information |
| `surname` | Personal data. Also 29.3% cardinality and no predictive value |
| `_ingested_at`, `_source_file`, `_source_url`, `_extracted_at` | Provenance. Stays in bronze, does not belong in a feature table |

### What is created

| Feature | Rule | Why |
|---|---|---|
| `balance_zero` | `balance == 0` | Section 2: a distinct state, not a missing value |
| `age_group` | 18-29 / 30-39 / 40-49 / 50-59 / 60+ | Lets linear models bend where risk is non-monotonic |
| `products_group` | 1 / 2 / 3+ | Collapses the two tiny high-risk bands into one usable category |
| `credit_score_band` | poor / fair / good / very good / excellent | Standard banking bands, readable by the business |

All four are **deterministic**: they depend only on the row itself, never on a
statistic computed across rows. That is what makes it safe to build them before
the split.

In [11]:
silver = (
    bronze
    .drop("row_number", "surname", "_ingested_at", "_source_file",
          "_source_url", "_extracted_at")

    # 0/1 integers to proper booleans
    .withColumn("has_cr_card",      F.col("has_cr_card")      == 1)
    .withColumn("is_active_member", F.col("is_active_member") == 1)
    .withColumn("exited",           F.col("exited")           == 1)
    .withColumn("balance",          F.col("balance").cast("double"))
    .withColumn("estimated_salary", F.col("estimated_salary").cast("double"))

    # deterministic features
    .withColumn("balance_zero", F.col("balance") == 0)
    .withColumn("age_group",
        F.when(F.col("age") < 30, "18-29")
         .when(F.col("age") < 40, "30-39")
         .when(F.col("age") < 50, "40-49")
         .when(F.col("age") < 60, "50-59")
         .otherwise("60+"))
    .withColumn("products_group",
        F.when(F.col("num_of_products") == 1, "1")
         .when(F.col("num_of_products") == 2, "2")
         .otherwise("3+"))
    .withColumn("credit_score_band",
        F.when(F.col("credit_score") < 580, "poor")
         .when(F.col("credit_score") < 670, "fair")
         .when(F.col("credit_score") < 740, "good")
         .when(F.col("credit_score") < 800, "very_good")
         .otherwise("excellent"))
)

print("columns in silver:", len(silver.columns))

columns in silver: 16


### Validate before writing

If any assertion fails nothing is persisted. A silver table that is silently wrong
is worse than no silver table: everything downstream would inherit the error and
none of it would report a problem.

In [12]:
s = silver

assert s.count() == 10000, "rows were lost"
assert s.select("customer_id").distinct().count() == 10000, "customer_id is no longer unique"
assert "surname" not in s.columns,    "surname propagated into silver"
assert "row_number" not in s.columns, "row_number propagated into silver"

nulls = (s.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in s.columns])
          .toPandas().T.rename(columns={0: "n_null"}))
assert nulls["n_null"].sum() == 0, f"nulls appeared:\n{nulls[nulls.n_null > 0]}"

for col, expected in [("age_group", 5), ("products_group", 3), ("credit_score_band", 5)]:
    n = s.select(col).distinct().count()
    print(f"  {col:<20} {n} categories (expected <= {expected})")

print("\nall validations passed")

  age_group            5 categories (expected <= 5)
  products_group       3 categories (expected <= 3)
  credit_score_band    5 categories (expected <= 5)

all validations passed


In [13]:
(silver.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(SILVER_TABLE))

spark.sql(f"""
    COMMENT ON TABLE {SILVER_TABLE} IS
    'Silver. Typed and clean from bronze, with deterministic features.
     No imputation, encoding or scaling: anything that learns statistics lives in
     the sklearn Pipeline in notebook 04, fitted after the split, to avoid leakage.'
""")

print("silver written:", SILVER_TABLE)
spark.table(SILVER_TABLE).limit(5).toPandas()

silver written: bank_churn_eng.silver.customers_clean


,customer_id,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited,balance_zero,age_group,products_group,credit_score_band
0,15634602,619,France,Female,42,2,0.00,1,True,True,101348.88,True,True,40-49,1,fair
1,15647311,608,Spain,Female,41,1,83807.86,1,False,True,112542.58,False,False,40-49,1,fair
2,15619304,502,France,Female,42,8,159660.80,3,True,False,113931.57,True,False,40-49,3+,poor
3,15701354,699,France,Female,39,1,0.00,2,False,False,93826.63,False,True,30-39,2,good
4,15737888,850,Spain,Female,43,2,125510.82,1,True,True,79084.10,False,False,40-49,1,excellent


## 6 · Quality profile — the "after"

The comparison against section 1.

In [14]:
sdf  = spark.table(SILVER_TABLE).toPandas()
cols = [c for c in sdf.columns if not c.startswith("_")]

profile_after = pd.DataFrame({
    "dtype":    sdf[cols].dtypes.astype(str),
    "n_null":   sdf[cols].isna().sum(),
    "n_unique": sdf[cols].nunique(),
})

print("BEFORE :", len(profile_before), "columns")
print("AFTER  :", len(profile_after), "columns")
print("\ndropped :", sorted(set(profile_before.index) - set(profile_after.index)))
print("created :", sorted(set(profile_after.index) - set(profile_before.index)))
profile_after

BEFORE : 14 columns
AFTER  : 16 columns

dropped : ['row_number', 'surname']
created : ['age_group', 'balance_zero', 'credit_score_band', 'products_group']


,dtype,n_null,n_unique
customer_id,int32,0,10000
credit_score,int32,0,460
geography,object,0,3
gender,object,0,2
age,int32,0,70
tenure,int32,0,11
balance,float64,0,6382
num_of_products,int32,0,4
has_cr_card,bool,0,2
is_active_member,bool,0,2


In [15]:
# No new category should be left underpopulated: a band with twelve customers
# would produce a coefficient that is pure noise.
for col in ["age_group", "products_group", "credit_score_band", "balance_zero"]:
    t = (sdf.groupby(col)
            .agg(n=("exited", "size"),
                 churn=("exited", lambda s: round(s.mean()*100, 1)))
            .sort_values("n", ascending=False))
    print(f"-- {col} --")
    print(t.to_string(), "\n")

-- age_group --
              n  churn
age_group             
30-39      4346   10.9
40-49      2618   30.8
18-29      1641    7.6
50-59       869   56.0
60+         526   27.9 

-- products_group --
                   n  churn
products_group             
1               5084   27.7
2               4590    7.6
3+               326   85.9 

-- credit_score_band --
                      n  churn
credit_score_band             
fair               3331   20.6
good               2428   18.6
poor               2362   22.0
very_good          1224   20.6
excellent           655   19.5 

-- balance_zero --
                 n  churn
balance_zero             
False         6383   24.1
True          3617   13.8 



## Result

```
bronze.customers_raw  ──▶  silver.customers_clean
   untouched copy             typed, clean, four derived features
```

**Decisions made and argued:** `balance = 0` kept as a legitimate state with a flag
rather than imputed; `surname` and `row_number` dropped; four deterministic features
created; the sensitive-variable decision deliberately deferred to notebook 05,
where it can be made against a measured cost.

**Findings carried forward:** the Germany confounder that halved the balance effect,
the gender gap that no observable variable explains, and sixty customers with four
products who all left.

**Next:** `00_problem_definition` on the first pass — the business baseline needs
silver to exist. Then `03_eda_analysis`.